<a href="https://colab.research.google.com/github/sidneiclementenobre/-p0-triador-feedbacks-gemini/blob/main/P0_Triador_Feedback_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto P0 — Triador de Feedbacks com IA (Google Gemini)

**Problema resolvido:** Classificar avaliações de clientes em categorias de sentimento (Positivo, Neutro ou Negativo) e extrair os pontos críticos mencionados de forma estruturada.

### Como configurar e executar este projeto:
1. No menu lateral esquerdo do Google Colab, clique no ícone de **Chave (Secrets)**.
2. Adicione um novo segredo chamado `GEMINI_API_KEY` com a sua chave gratuita do Google AI Studio.
3. Ative a permissão de acesso ao notebook na chave seletora.
4. Execute todas as células em ordem (`Ambiente de execução > Executar tudo`).


In [23]:
!pip install -q google-genai pydantic


In [31]:
import os
from typing import List
from google import genai
from google.genai import types
from google.genai.errors import APIError
from google.colab import userdata
from pydantic import BaseModel, Field

class FeedbackAnalise(BaseModel):
    sentimento: str = Field(
        description="Deve ser estritamente: Positivo, Neutro ou Negativo"
    )
    pontos_criticos: List[str] = Field(
        description="Lista objetiva dos problemas, elogios ou pontos importantes citados"
    )
    resumo_executivo: str = Field(
        description="Um resumo de uma única frase sobre o feedback do cliente"
    )

def analisar_feedback(texto_cliente: str) -> FeedbackAnalise:
    try:
        api_key = userdata.get("GEMINI_API_KEY")
    except userdata.SecretNotFoundError:
        raise ValueError("Erro: Segredo 'GEMINI_API_KEY' não encontrado. Configure no ícone de chave à esquerda.")
    except userdata.NotebookAccessError:
        raise PermissionError("Erro: Ative a permissão de acesso ao segredo 'GEMINI_API_KEY' no painel lateral.")

    if not texto_cliente or not texto_cliente.strip():
        raise ValueError("Erro: O texto do feedback não pode estar vazio.")

    client = genai.Client(api_key=api_key, http_options={"api_version": "v1beta"})

    prompt_sistema = (
        "Você é um especialista em atendimento ao cliente e análise de dados. "
        "Leia o feedback do cliente, classifique o sentimento predominante e "
        "extraia os pontos críticos de forma clara e objetiva."
    )

    # Lista dos modelos atuais da família Gemini 3 liberados para chaves novas
    modelos_atuais = [
        "gemini-3-flash-preview",
        "gemini-3.1-flash-lite-preview",
        "gemini-3-pro-preview"
    ]

    ultimo_erro = None
    for modelo in modelos_atuais:
        try:
            response = client.models.generate_content(
                model=modelo,
                contents=texto_cliente,
                config=types.GenerateContentConfig(
                    system_instruction=prompt_sistema,
                    response_mime_type="application/json",
                    response_schema=FeedbackAnalise,
                    temperature=0.1,
                ),
            )
            print(f"[Modelo utilizado com sucesso: {modelo}]")
            return FeedbackAnalise.model_validate_json(response.text)

        except APIError as e:
            ultimo_erro = e
            continue

    raise RuntimeError(f"Nenhum modelo respondeu. Último erro da API: {ultimo_erro}")


In [32]:
def executar_fluxo(texto: str):
    print("--- Iniciando Análise de Feedback com IA ---")
    print(f'Texto original:\n"{texto}"\n')
    print("Processando no Google Gemini...")

    try:
        resultado = analisar_feedback(texto)

        print("\n--- Resultado da Análise Estruturada ---")
        print(f"Sentimento predominante: {resultado.sentimento}")
        print("Pontos críticos identificados:")
        for ponto in resultado.pontos_criticos:
            print(f"  - {ponto}")
        print(f"Resumo da IA: {resultado.resumo_executivo}")

    except Exception as e:
        print(f"\nFalha na execução: {e}")

# Teste 1: Feedback misto/negativo
feedback_teste = (
    "Adorei a qualidade da comida e o sabor estava incrível! "
    "Porém, o motoboy demorou mais de 1 hora para entregar e a batata frita "
    "chegou totalmente murcha e fria. Fiquei bem chateado com o atraso."
)

executar_fluxo(feedback_teste)


--- Iniciando Análise de Feedback com IA ---
Texto original:
"Adorei a qualidade da comida e o sabor estava incrível! Porém, o motoboy demorou mais de 1 hora para entregar e a batata frita chegou totalmente murcha e fria. Fiquei bem chateado com o atraso."

Processando no Google Gemini...
[Modelo utilizado com sucesso: gemini-3-flash-preview]

--- Resultado da Análise Estruturada ---
Sentimento predominante: Negativo
Pontos críticos identificados:
  - Qualidade e sabor da comida excelentes
  - Atraso na entrega superior a 1 hora
  - Batata frita murcha e fria
  - Insatisfação com o serviço de entrega
Resumo da IA: O cliente apreciou o sabor da comida, mas manifestou insatisfação devido ao longo atraso na entrega e ao estado inadequado das batatas fritas.
